## Cell 1 — Hardware Check
Detects available hardware. TPU is fastest, T4 GPU works fine (~2-5 min per generation). CPU will be very slow and is not recommended.

In [ ]:
import jax

# v5e and newer TPUs need explicit initialization before JAX can detect them
try:
    jax.distributed.initialize()
except Exception:
    pass  # already initialized or not needed on this runtime

try:
    tpu_devices = jax.devices('tpu')
    DEVICE = 'tpu'
    print(f'TPU detected: {tpu_devices}')
except RuntimeError:
    try:
        gpu_devices = jax.devices('gpu')
        DEVICE = 'gpu'
        print(f'GPU detected: {gpu_devices}')
        print('Note: GPU generation is slower than TPU (~2-5 min per track).')
    except RuntimeError:
        raise RuntimeError(
            'Only CPU detected. Generation will be extremely slow.\n'
            'Go to Runtime → Change runtime type and select T4 GPU or v5e-1 TPU.'
        )

## Cell 2 — Install Dependencies
Installs Magenta RT, Flask, and ngrok. Takes about a minute.

In [ ]:
!pip install -q magenta-rt flask flask-cors pyngrok scipy
print('Dependencies installed.')

## Cell 3 — Config
**This is the only cell you need to edit.** Set your ngrok auth token (free at ngrok.com → Your Authtoken).

In [ ]:
NGROK_AUTH_TOKEN = 'your_ngrok_token_here'  # get free token at ngrok.com
DEFAULT_DURATION_SECONDS = 30

## Cell 4 — Load Magenta RT Model
Downloads and loads the model. Takes 1–2 minutes. You'll see a confirmation when it's ready.

In [ ]:
import magenta_rt

try:
    model = magenta_rt.MagentaRT(device=DEVICE)
    print(f'Magenta RT model loaded on {DEVICE.upper()} and ready.')
except Exception as e:
    raise RuntimeError(f'Failed to load Magenta RT model: {e}')

## Cell 5 — Start Generation Server
Starts a Flask server inside Colab and exposes it publicly via ngrok.

**After running this cell, copy the URL printed below and paste it into the Electron app.**

This cell blocks — leave it running while you use the app. To stop the server, interrupt the cell.

In [ ]:
from flask import Flask, request, send_file
from flask_cors import CORS
from pyngrok import ngrok
import numpy as np
import scipy.io.wavfile
import io

# Step 7b: Flask server running inside Colab, exposed via ngrok
server = Flask(__name__)
CORS(server)

@server.route('/health')
def health():
    return {'status': 'ok'}

@server.route('/generate', methods=['POST'])
def generate():
    data = request.get_json()
    prompt = data.get('prompt', '')
    duration_seconds = data.get('duration_seconds', DEFAULT_DURATION_SECONDS)

    if not prompt:
        return {'error': 'prompt is required'}, 400

    print(f'Generating: "{prompt}" for {duration_seconds}s')

    num_chunks = max(1, int(duration_seconds / 2))
    chunks = []
    for i in range(num_chunks):
        chunk = model.generate_chunk(prompt)
        chunks.append(chunk)
        print(f'  chunk {i+1}/{num_chunks}')

    audio = np.concatenate(chunks, axis=-1)

    buf = io.BytesIO()
    scipy.io.wavfile.write(buf, rate=48000, data=audio.T)
    buf.seek(0)

    print('Generation complete. Sending WAV.')
    return send_file(buf, mimetype='audio/wav', download_name='accompaniment.wav')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000).public_url
print(f'\n===============================')
print(f'Paste this URL into the app:')
print(f'{public_url}')
print(f'===============================')

server.run(port=5000)